#  Part 2: The Generative AI World

## From Predictive to Generative AI


### What You'll Learn
- How Generative AI solves the same problem using **natural language** instead of numbers
- **Prompt engineering** : the new programming paradigm
- **Local AI on Ubuntu** with Ollama (privacy-first, no cloud needed)
- **Cloud AI** with Google Gemini (fast, powerful, API-based)
- Building **robust AI systems** with fallbacks

### The Paradigm Shift
> *In Part 1, we fed numbers to an algorithm and got back 0 or 1.*
>
> *Now, we'll have a conversation with AI in plain English and get explanations, insights and creativity.*

### Why Ubuntu?
Ubuntu is the #1 platform for AI development. Ollama runs natively on Linux, giving you **private, local AI** without sending data to any cloud.

---

## Setup: Install & Import Libraries

We need three things:
- **requests** : to talk to Ollama's local API
- **google-generativeai** : to talk to Gemini's cloud API
- **pandas** : to load and analyze the Titanic data

In [1]:
#!pip install -q google-generativeai requests pandas

import requests
import json
import pandas as pd
from typing import Dict

# We'll import Gemini only when needed — Ollama needs no special library
try:
    import google.generativeai as genai
    GENAI_AVAILABLE = True
except ImportError:
    GENAI_AVAILABLE = False

print(' Libraries loaded')
print(f'   Ollama: uses requests (always available)')
print(f'   Gemini: {"ready" if GENAI_AVAILABLE else "not installed"}')
print()
print(' Ready to explore Generative AI')

 Libraries loaded
   Ollama: uses requests (always available)
   Gemini: ready

 Ready to explore Generative AI


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Prompt Engineering : The New Programming


In traditional ML, we wrote **code** to process data.
In Generative AI, we write **prompts** to instruct the model.

> *"The quality of your prompt = the quality of your result."*

### Anatomy of a Good Prompt

| Component | Purpose | Example |
|-----------|---------|--------|
| **Role** | Tell the AI *who* it is | "You are a Titanic historian" |
| **Context** | Give background information | "891 passengers, 38% survived" |
| **Task** | What you want it to do | "Predict survival" |
| **Format** | How to structure the answer | "Reply with SURVIVED or DIED" |

### Key Principle: Start Simple
Complex prompts often confuse models. We'll start minimal and build up.

In [3]:
# --- Prompt Templates ---
# We'll use these with BOTH Ollama and Gemini

# Simple prompt — clear, direct, easy for any model to follow
SIMPLE_PROMPT = """Titanic passenger: {gender}, age {age}, class {pclass}, fare ${fare}

Predict survival. Reply with just:
SURVIVED or DIED"""

# Detailed prompt — adds role and reasoning
DETAILED_PROMPT = """You are a Titanic historian and data analyst.

Passenger profile:
- Gender: {gender}
- Age: {age}
- Class: {pclass}
- Fare: ${fare}

Based on historical survival patterns, predict this passenger's fate.
Reply in this exact format:
Prediction: SURVIVED or DIED
Reason: one sentence explanation"""

# Let's see what these look like with real data
sample = {'gender': 'Female', 'age': 25, 'pclass': 1, 'fare': 100}

print(' SIMPLE PROMPT:')
print('-' * 40)
print(SIMPLE_PROMPT.format(**sample))
print()
print(' DETAILED PROMPT:')
print('-' * 40)
print(DETAILED_PROMPT.format(**sample))
print()
print(' Same data, different instructions, the AI will respond differently to each.')

 SIMPLE PROMPT:
----------------------------------------
Titanic passenger: Female, age 25, class 1, fare $100

Predict survival. Reply with just:
SURVIVED or DIED

 DETAILED PROMPT:
----------------------------------------
You are a Titanic historian and data analyst.

Passenger profile:
- Gender: Female
- Age: 25
- Class: 1
- Fare: $100

Based on historical survival patterns, predict this passenger's fate.
Reply in this exact format:
Prediction: SURVIVED or DIED
Reason: one sentence explanation

 Same data, different instructions, the AI will respond differently to each.


In [4]:
# --- Fallback: Rule-Based Prediction ---
# Always have a backup when AI is unavailable or returns garbage

def rule_based_prediction(passenger: Dict) -> Dict:
    """Simple historical rules as a safety net."""
    if passenger['gender'] == 'Female':
        if passenger['pclass'] <= 2:
            return {'prediction': 'SURVIVED', 'confidence': 0.92, 'reasoning': 'Upper-class women had ~92% survival'}
        else:
            return {'prediction': 'SURVIVED', 'confidence': 0.50, 'reasoning': '3rd-class women had ~50% survival'}
    else:
        if passenger['pclass'] == 1 and passenger['age'] < 16:
            return {'prediction': 'SURVIVED', 'confidence': 0.60, 'reasoning': '1st-class boys had reasonable survival odds'}
        elif passenger['pclass'] == 1:
            return {'prediction': 'DIED', 'confidence': 0.63, 'reasoning': '1st-class men had ~37% survival'}
        else:
            return {'prediction': 'DIED', 'confidence': 0.86, 'reasoning': 'Lower-class men had very low survival (~14%)'}

# Test it
test = {'gender': 'Female', 'age': 25, 'pclass': 1, 'fare': 100}
result = rule_based_prediction(test)
print(f' Rule-based fallback test:')
print(f'   Passenger: {test["gender"]}, Age {test["age"]}, Class {test["pclass"]}')
print(f'   → {result["prediction"]} ({result["confidence"]:.0%}) — {result["reasoning"]}')
print()
print(' Fallback ready, this runs even without internet or AI models.')

 Rule-based fallback test:
   Passenger: Female, Age 25, Class 1
   → SURVIVED (92%) — Upper-class women had ~92% survival

 Fallback ready, this runs even without internet or AI models.


## Method 1: Ollama : Local AI on Ubuntu


[Ollama](https://ollama.ai) lets you run large language models **entirely on your machine**.

### Why Local AI Matters
-  **Privacy**: Your data never leaves your computer
-  **Free**: No API costs, no usage limits
-  **Offline**: Works without internet
-  **Native Linux**: Ollama runs best on Ubuntu/Linux

### Quick Setup (Ubuntu Terminal)
```bash
# Install Ollama (one command)
curl -fsSL https://ollama.ai/install.sh | sh

# Download a model
ollama pull llama3

# Ollama server starts automatically on install
# Verify it's running:
curl http://localhost:11434/api/tags
```

Ollama exposes a simple REST API on `localhost:11434` — no SDK needed, just HTTP requests.

In [5]:
# --- Detect Ollama & Select Model ---

OLLAMA_URL = 'http://localhost:11434'
ollama_ready = False
ollama_model = None

try:
    print(' Checking for Ollama server...')
    resp = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)

    if resp.status_code == 200:
        models = resp.json().get('models', [])
        model_names = [m['name'] for m in models]
        print(f' Ollama is running. Models available: {len(models)}')

        if model_names:
            for name in model_names:
                print(f'   • {name}')

        # Prefer llama3, fall back to any available model
        llama3 = [n for n in model_names if 'llama3' in n.lower()]
        if llama3:
            ollama_model = llama3[0]
        elif model_names:
            ollama_model = model_names[0]

        if ollama_model:
            ollama_ready = True
            print(f'\n Selected model: {ollama_model}')
        else:
            print('\n  No models found. Run: ollama pull llama3')
    else:
        print(f' Ollama returned status {resp.status_code}')

except requests.exceptions.ConnectionError:
    print(' Ollama not running')
    print()
    print(' This is expected in Google Colab (Ollama is local only).')
    print('   On Ubuntu, install with: curl -fsSL https://ollama.ai/install.sh | sh')
    print('   Then run: ollama pull llama3')

print(f'\n🧪 Ollama ready: {ollama_ready}')

 Checking for Ollama server...
 Ollama not running

 This is expected in Google Colab (Ollama is local only).
   On Ubuntu, install with: curl -fsSL https://ollama.ai/install.sh | sh
   Then run: ollama pull llama3

🧪 Ollama ready: False


In [ ]:
# --- Predict with Ollama ---

def predict_with_ollama(passenger: Dict, prompt_template=SIMPLE_PROMPT) -> Dict:
    """Send a prompt to Ollama and parse the prediction."""
    try:
        prompt = prompt_template.format(
            gender=passenger['gender'],
            age=passenger['age'],
            pclass=passenger['pclass'],
            fare=passenger['fare']
        )

        resp = requests.post(
            f'{OLLAMA_URL}/api/generate',
            json={'model': ollama_model, 'prompt': prompt, 'stream': False, 'options': {'temperature': 0.1}},
            timeout=30
        )

        if resp.status_code != 200:
            raise Exception(f'API error {resp.status_code}')

        text = resp.json().get('response', '').strip()
        upper = text.upper()

        # Parse response
        if 'SURVIVED' in upper:
            prediction = 'SURVIVED'
        elif 'DIED' in upper:
            prediction = 'DIED'
        else:
            # AI gave an unclear answer — fall back to rules
            fallback = rule_based_prediction(passenger)
            fallback['reasoning'] = f'Fallback (Ollama said: "{text[:80]}...")'
            return fallback

        return {
            'prediction': prediction,
            'confidence': 0.80,
            'reasoning': text[:200]
        }

    except Exception as e:
        print(f'   ⚠️  Ollama error: {e}')
        return rule_based_prediction(passenger)

print('✅ Ollama prediction function defined')

✅ Ollama prediction function defined


In [ ]:
# --- Test Ollama ---

if ollama_ready:
    print('🧪 Testing Ollama with two passengers:\n')

    test_passengers = [
        {'gender': 'Female', 'age': 25, 'pclass': 1, 'fare': 100},
        {'gender': 'Male',   'age': 30, 'pclass': 3, 'fare': 15},
    ]

    for p in test_passengers:
        print(f'🧑 {p["gender"]}, Age {p["age"]}, Class {p["pclass"]}')

        # Simple prompt
        r1 = predict_with_ollama(p, SIMPLE_PROMPT)
        print(f'   Simple  → {r1["prediction"]} ({r1["confidence"]:.0%})')

        # Detailed prompt
        r2 = predict_with_ollama(p, DETAILED_PROMPT)
        print(f'   Detailed → {r2["prediction"]} ({r2["confidence"]:.0%})')
        print(f'   Reasoning: {r2["reasoning"][:120]}')
        print()

    print('💡 Notice: The detailed prompt gives us an EXPLANATION — something Predictive AI could never do!')
else:
    print('⏭️  Ollama not available — skipping test.')
    print('   We\'ll use Gemini (cloud) or the rule-based fallback instead.')

🧪 Testing Ollama with two passengers:

🧑 Female, Age 25, Class 1
   Simple  → DIED (80%)
   Detailed → DIED (80%)
   Reasoning: Prediction: DIED
Reason: Given the relatively low fare and class of the passenger, it is likely that she was traveling w

🧑 Male, Age 30, Class 3
   Simple  → DIED (80%)
   Detailed → DIED (80%)
   Reasoning: Prediction: DIED
Reason: Given the low fare and third-class status, it is likely that this passenger was among the most 

💡 Notice: The detailed prompt gives us an EXPLANATION — something Predictive AI could never do!


## Method 2: Google Gemini — Cloud AI ☁️

---

[Google Gemini](https://ai.google.dev/) is a powerful cloud-based LLM accessible via API.

### Cloud vs Local — The Trade-off

| | Ollama (Local) | Gemini (Cloud) |
|---|---|---|
| **Privacy** | ✅ Data stays on your machine | ⚠️ Data sent to Google |
| **Cost** | ✅ Free forever | ⚠️ Free tier, then pay-per-request |
| **Speed** | Depends on your hardware | ✅ Fast (Google's servers) |
| **Model quality** | Good (llama3, mistral) | ✅ State-of-the-art |
| **Internet** | ✅ Works offline | ❌ Requires connection |
| **Setup** | Install Ollama + model | ❌ Need API key |

### Getting a Free API Key
1. Go to [Google AI Studio](https://makersuite.google.com/)
2. Sign in with your Google account
3. Click **Get API Key** → **Create API Key**
4. Copy the key and paste it below

> ⚠️ **Security**: Never commit API keys to version control. Use environment variables in production.

In [ ]:
# --- Setup Gemini ---

gemini_ready = False
gemini_model = None

if not GENAI_AVAILABLE:
    print('❌ google-generativeai not installed. Run: pip install google-generativeai')
else:
    print('🔑 Gemini API Setup')
    print('-' * 40)

    # ⚠️ WORKSHOP DEMO ONLY — replace with your key
    # In production, use: import os; api_key = os.environ['GEMINI_API_KEY']
    api_key = ''

    if api_key == 'YOUR_API_KEY_HERE':
        print('⚠️  No API key set.')
        print('   Get a free key: https://makersuite.google.com/')
        print('   Then replace YOUR_API_KEY_HERE above.')
    else:
        try:
            genai.configure(api_key=api_key)
            gemini_model = genai.GenerativeModel('gemini-2.0-flash')
            gemini_ready = True
            print('✅ Gemini configured successfully!')
            print(f'   Model: gemini-2.0-flash')
        except Exception as e:
            print(f'❌ Gemini setup failed: {e}')

print(f'\n🧪 Gemini ready: {gemini_ready}')

🔑 Gemini API Setup
----------------------------------------
✅ Gemini configured successfully!
   Model: gemini-2.0-flash

🧪 Gemini ready: True


In [ ]:
# --- Predict with Gemini ---

def predict_with_gemini(passenger: Dict, prompt_template=SIMPLE_PROMPT) -> Dict:
    """Send a prompt to Gemini and parse the prediction."""
    try:
        prompt = prompt_template.format(
            gender=passenger['gender'],
            age=passenger['age'],
            pclass=passenger['pclass'],
            fare=passenger['fare']
        )

        response = gemini_model.generate_content(prompt)
        text = response.text.strip()
        upper = text.upper()

        # Parse response
        if 'SURVIVED' in upper:
            prediction = 'SURVIVED'
        elif 'DIED' in upper:
            prediction = 'DIED'
        else:
            fallback = rule_based_prediction(passenger)
            fallback['reasoning'] = f'Fallback (Gemini said: "{text[:80]}...")'
            return fallback

        return {
            'prediction': prediction,
            'confidence': 0.80,
            'reasoning': text[:200]
        }

    except Exception as e:
        print(f'   ⚠️  Gemini error: {e}')
        return rule_based_prediction(passenger)

print('✅ Gemini prediction function defined')

✅ Gemini prediction function defined


In [ ]:
# --- Test Gemini ---

if gemini_ready:
    print('🧪 Testing Gemini with two passengers:\n')

    test_passengers = [
        {'gender': 'Female', 'age': 25, 'pclass': 1, 'fare': 100},
        {'gender': 'Male',   'age': 30, 'pclass': 3, 'fare': 15},
    ]

    for p in test_passengers:
        print(f'🧑 {p["gender"]}, Age {p["age"]}, Class {p["pclass"]}')

        # Simple prompt
        r1 = predict_with_gemini(p, SIMPLE_PROMPT)
        print(f'   Simple  → {r1["prediction"]} ({r1["confidence"]:.0%})')

        # Detailed prompt
        r2 = predict_with_gemini(p, DETAILED_PROMPT)
        print(f'   Detailed → {r2["prediction"]} ({r2["confidence"]:.0%})')
        print(f'   Reasoning: {r2["reasoning"][:120]}')
        print()

    print('💡 Same prompts, different AI engine — compare these results with Ollama above!')
else:
    print('⏭️  Gemini not available — skipping test.')
    print('   Add your API key above, or use Ollama/fallback instead.')

🧪 Testing Gemini with two passengers:

🧑 Female, Age 25, Class 1
   ⚠️  Gemini error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 25.123248642s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  

## The AI Showdown: Ollama vs Gemini vs Rules

---

Now the fun part. Let's run **all three methods** on the same passengers and compare.

This is the core lesson: **same problem, three different approaches, different trade-offs.**

In [ ]:
# --- The Showdown Passengers ---

showdown_passengers = [
    {'name': 'Lady Margaret',  'gender': 'Female', 'age': 25, 'pclass': 1, 'fare': 100,
     'desc': '1st Class Woman, 25'},
    {'name': 'John Smith',     'gender': 'Male',   'age': 30, 'pclass': 3, 'fare': 15,
     'desc': '3rd Class Man, 30'},
    {'name': 'Mrs. Brown',     'gender': 'Female', 'age': 45, 'pclass': 2, 'fare': 50,
     'desc': '2nd Class Woman, 45'},
    {'name': 'Young Tommy',    'gender': 'Male',   'age': 8,  'pclass': 3, 'fare': 20,
     'desc': '3rd Class Boy, 8'},
    {'name': 'Lord Ashton',    'gender': 'Male',   'age': 55, 'pclass': 1, 'fare': 250,
     'desc': '1st Class Man, 55'},
]

print(f'🎭 {len(showdown_passengers)} passengers ready for the showdown!')
for p in showdown_passengers:
    print(f'   • {p["name"]} — {p["desc"]}')

🎭 5 passengers ready for the showdown!
   • Lady Margaret — 1st Class Woman, 25
   • John Smith — 3rd Class Man, 30
   • Mrs. Brown — 2nd Class Woman, 45
   • Young Tommy — 3rd Class Boy, 8
   • Lord Ashton — 1st Class Man, 55


In [ ]:
# --- Run the Showdown ---

print('🥊 AI SHOWDOWN: Ollama vs Gemini vs Rules')
print('=' * 70)

results = []

for p in showdown_passengers:
    print(f'\n👤 {p["name"]} ({p["desc"]})')
    print(f'   {p["gender"]}, Age {p["age"]}, Class {p["pclass"]}, Fare ${p["fare"]}')

    row = {'Passenger': f'{p["name"]} ({p["desc"]})'}

    # Ollama
    if ollama_ready:
        r = predict_with_ollama(p)
        row['Ollama'] = f'{r["prediction"]} ({r["confidence"]:.0%})'
        print(f'   🦙 Ollama: {row["Ollama"]}')
    else:
        row['Ollama'] = 'N/A'

    # Gemini
    if gemini_ready:
        r = predict_with_gemini(p)
        row['Gemini'] = f'{r["prediction"]} ({r["confidence"]:.0%})'
        print(f'   🤖 Gemini: {row["Gemini"]}')
    else:
        row['Gemini'] = 'N/A'

    # Rules
    r = rule_based_prediction(p)
    row['Rules'] = f'{r["prediction"]} ({r["confidence"]:.0%})'
    print(f'   📏 Rules:  {row["Rules"]}')

    results.append(row)

# Summary table
print('\n' + '=' * 70)
print('📊 FINAL COMPARISON TABLE')
print('=' * 70)
print(pd.DataFrame(results).to_string(index=False))

🥊 AI SHOWDOWN: Ollama vs Gemini vs Rules

👤 Lady Margaret (1st Class Woman, 25)
   Female, Age 25, Class 1, Fare $100
   🦙 Ollama: DIED (80%)
   ⚠️  Gemini error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 55.203061756s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metri

### 💡 What Did We Just See?

| Observation | Lesson |
|---|---|
| AI models sometimes **disagree** | No single AI is always right |
| Rules are **instant and consistent** | Simple solutions have value |
| AI can **explain its reasoning** | This is what Predictive AI can't do |
| Different prompts give **different results** | Prompt engineering matters |
| Local AI works **without internet** | Privacy and reliability |

> **The best AI systems combine multiple approaches.** Use AI for reasoning, rules for reliability, and always have a fallback.

## Interactive Chat: Ask AI About the Data 💬

---

This is where Generative AI truly shines over Predictive AI.

Instead of just getting 0 or 1, we can **have a conversation** with the data:
- Ask *why* something happened
- Get insights in plain English
- Explore "what if" scenarios
- Use AI as an **analysis partner**, not just a calculator

### How It Works
1. We load the real Titanic dataset and compute actual statistics
2. For data questions, we answer with **real numbers first**
3. For broader questions, we ask the AI using its **knowledge**
4. If everything fails, we have **hardcoded fallback answers**

> *Three layers of reliability: Data → AI → Fallback*

In [ ]:
# --- Load Titanic Data for Chat ---

try:
    url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    titanic_df = pd.read_csv(url)
    print(f'📊 Titanic data loaded: {titanic_df.shape[0]} passengers')
except Exception:
    titanic_df = None
    print('⚠️  Could not load data — chat will use AI knowledge only.')

def get_titanic_stats():
    """Compute real statistics from the dataset."""
    if titanic_df is None:
        return None
    df = titanic_df
    age_df = df.dropna(subset=['Age'])
    return {
        'total':     df.shape[0],
        'survived':  int(df['Survived'].sum()),
        'overall':   df['Survived'].mean(),
        'female':    df[df['Sex'] == 'female']['Survived'].mean(),
        'male':      df[df['Sex'] == 'male']['Survived'].mean(),
        'class_1':   df[df['Pclass'] == 1]['Survived'].mean(),
        'class_2':   df[df['Pclass'] == 2]['Survived'].mean(),
        'class_3':   df[df['Pclass'] == 3]['Survived'].mean(),
        'children':  age_df[age_df['Age'] < 18]['Survived'].mean(),
        'adults':    age_df[age_df['Age'] >= 18]['Survived'].mean(),
        'avg_age':   age_df['Age'].mean(),
        'avg_fare':  df['Fare'].mean(),
    }

stats = get_titanic_stats()
if stats:
    print(f'   Overall survival: {stats["overall"]:.1%}')
    print(f'   Women: {stats["female"]:.1%} | Men: {stats["male"]:.1%}')
    print(f'   1st: {stats["class_1"]:.1%} | 2nd: {stats["class_2"]:.1%} | 3rd: {stats["class_3"]:.1%}')
    print(f'   Children: {stats["children"]:.1%} | Adults: {stats["adults"]:.1%}')
print()
print('✅ Data analysis layer ready')

📊 Titanic data loaded: 891 passengers
   Overall survival: 38.4%
   Women: 74.2% | Men: 18.9%
   1st: 63.0% | 2nd: 47.3% | 3rd: 24.2%
   Children: 54.0% | Adults: 38.1%

✅ Data analysis layer ready


In [ ]:
# --- Data-First Answer Engine ---

def answer_from_data(question: str) -> str:
    """Try to answer using real dataset statistics."""
    if not stats:
        return None
    q = question.lower()

    if ('women' in q or 'female' in q) and ('men' in q or 'male' in q or 'gender' in q):
        return (f'📊 REAL DATA: Women had {stats["female"]:.1%} survival vs men at {stats["male"]:.1%}. '
                f'The "women and children first" policy was clearly enforced.')

    if 'class' in q and ('surviv' in q or 'affect' in q or 'impact' in q or 'rate' in q):
        return (f'📊 REAL DATA: 1st class: {stats["class_1"]:.1%}, 2nd class: {stats["class_2"]:.1%}, '
                f'3rd class: {stats["class_3"]:.1%}. Higher class = closer to lifeboats.')

    if 'age' in q or 'children' in q or 'child' in q or 'kid' in q:
        return (f'📊 REAL DATA: Children (<18): {stats["children"]:.1%} survival vs adults: {stats["adults"]:.1%}. '
                f'Average age: {stats["avg_age"]:.0f} years.')

    if 'survival rate' in q or 'overall' in q or 'how many' in q:
        return (f'📊 REAL DATA: {stats["overall"]:.1%} overall survival rate '
                f'({stats["survived"]}/{stats["total"]} passengers).')

    if 'fare' in q or 'ticket' in q or 'price' in q:
        return f'📊 REAL DATA: Average fare was £{stats["avg_fare"]:.1f}. Higher fares strongly correlated with survival.'

    return None  # No data-driven answer available

print('✅ Data-first answer engine ready')

✅ Data-first answer engine ready


In [ ]:
# --- Unified Chat Function ---
# Tries: Data → Ollama → Gemini → Fallback

def chat(question: str) -> str:
    """Ask a question about the Titanic. Uses the best available source."""

    # Layer 1: Real data
    data_answer = answer_from_data(question)
    if data_answer:
        return data_answer

    # Layer 2: AI (prefer Ollama for privacy, fall back to Gemini)
    ai_prompt = f"""You are a Titanic historian and data expert.
Key facts: 891 passengers, 38% survived. Women: 74%, Men: 19%.
Class survival — 1st: 63%, 2nd: 47%, 3rd: 24%.

Question: {question}

Answer clearly and concisely:"""

    # Try Ollama first (local, private)
    if ollama_ready:
        try:
            resp = requests.post(
                f'{OLLAMA_URL}/api/generate',
                json={'model': ollama_model, 'prompt': ai_prompt, 'stream': False, 'options': {'temperature': 0.3}},
                timeout=30
            )
            if resp.status_code == 200:
                text = resp.json().get('response', '').strip()
                if text:
                    return f'🦙 Ollama: {text}'
        except Exception:
            pass

    # Try Gemini
    if gemini_ready:
        try:
            response = gemini_model.generate_content(ai_prompt)
            text = response.text.strip()
            if text:
                return f'🤖 Gemini: {text}'
        except Exception:
            pass

    # Layer 3: Hardcoded fallback
    return ('📚 Based on historical records: Survival depended primarily on gender, class, and age. '
            'Women and children in upper classes had the best chances.')

print('✅ Chat system ready!')
print()
print('Usage: chat("your question here")')
print()
print('Try these:')
print('  chat("Why did more women survive than men?")')
print('  chat("How did class affect survival?")')
print('  chat("What was the overall survival rate?")')
print('  chat("What lessons can we learn from the Titanic?")')

✅ Chat system ready!

Usage: chat("your question here")

Try these:
  chat("Why did more women survive than men?")
  chat("How did class affect survival?")
  chat("What was the overall survival rate?")
  chat("What lessons can we learn from the Titanic?")


In [ ]:
# --- Try It! Data-driven question ---

print('🗣️  Question: Why did more women survive than men?')
print()
print(chat('Why did more women survive than men?'))

🗣️  Question: Why did more women survive than men?

📊 REAL DATA: Women had 74.2% survival vs men at 18.9%. The "women and children first" policy was clearly enforced.


In [ ]:
# --- Try It! AI knowledge question ---

print('🗣️  Question: What lessons can we learn from the Titanic disaster?')
print()
print(chat('What lessons can we learn from the Titanic disaster?'))

🗣️  Question: What lessons can we learn from the Titanic disaster?

🦙 Ollama: The Titanic disaster provides valuable lessons for maritime safety, emergency preparedness, and social responsibility. Some key takeaways include:

1. **Improved Safety Protocols**: The tragedy highlighted the importance of robust safety measures, such as adequate lifeboat capacity, regular drills, and enhanced communication between crew members.
2. **Social Inequality and Discrimination**: The stark contrast in survival rates among passengers by class underscores the need for equal treatment and opportunities, regardless of social status or background.
3. **Human Error and Training**: The disaster was partly attributed to human error, inadequate training, and complacency, emphasizing the importance of rigorous crew training and continuous improvement.
4. **Risk Assessment and Preparedness**: The Titanic's builders and operators underestimated the risks associated with navigating the North Atlantic in ice-fil

In [ ]:
# --- Try It! Creative question ---

print('🗣️  Question: If you were on the Titanic, what would increase your survival chances?')
print()
print(chat('If you were on the Titanic, what would increase your survival chances?'))

🗣️  Question: If you were on the Titanic, what would increase your survival chances?

🦙 Ollama: To increase your survival chances on the Titanic:

1. **Be a woman**: Women had a significantly higher survival rate (74%) compared to men (19%).
2. **Be in 1st class**: First-class passengers had a much higher survival rate (63%) compared to second-class (47%) and third-class (24%).
3. **Be young or middle-aged**: The survival rates are not provided for specific age groups, but it's generally believed that younger adults had better chances of survival.
4. **Have access to lifeboats**: Being able to board a lifeboat would significantly increase your chances of survival.

Keep in mind that these factors were influenced by various social and economic factors at the time.


### 🎯 Your Turn! Ask Your Own Questions

Copy the cell below and try your own questions. Here are some ideas:

**Data questions** (answered from real statistics):
- `chat('How did class affect survival rates?')`
- `chat('What was the survival rate for children?')`
- `chat('How much did tickets cost?')`

**Knowledge questions** (answered by AI):
- `chat('What happened during the Titanic sinking?')`
- `chat('How did the ship's design affect evacuation?')`
- `chat('What safety changes came after the Titanic?')`

**Creative questions** (AI reasoning + creativity):
- `chat('Write a survival guide for 1912 passengers')`
- `chat('What questions would a journalist ask about this data?')`
- `chat('How would social media have changed the disaster?')`

In [ ]:
# --- Your question here! ---
print('🗣️  Question: How did class affect survival rates?')
print()
print(chat('How did class affect survival rates?'))

🗣️  Question: How did class affect survival rates?

📊 REAL DATA: 1st class: 63.0%, 2nd class: 47.3%, 3rd class: 24.2%. Higher class = closer to lifeboats.


## Prompt Engineering Playground 🧪


Now it's your turn to experiment. The exercises below show how **small changes to a prompt** produce **dramatically different results**.

This is the core skill of working with Generative AI.

In [ ]:
# --- Experiment 1: Role Changes Everything ---
# Same question, different roles — watch how the answer shifts

question = 'Why did so many people die on the Titanic?'

roles = [
    'You are a maritime safety engineer.',
    'You are a social historian studying class inequality.',
    'You are a 10-year-old explaining to a friend.',
]

print('🎭 EXPERIMENT 1: How Role Changes the Answer')
print(f'Question: {question}')
print('=' * 60)

for role in roles:
    prompt = f'{role}\n\n{question}\n\nAnswer in 2-3 sentences:'
    print(f'\n🎭 Role: {role}')

    if ollama_ready:
        try:
            resp = requests.post(
                f'{OLLAMA_URL}/api/generate',
                json={'model': ollama_model, 'prompt': prompt, 'stream': False, 'options': {'temperature': 0.3}},
                timeout=30
            )
            if resp.status_code == 200:
                print(f'   {resp.json().get("response", "").strip()[:300]}')
                continue
        except Exception:
            pass

    if gemini_ready:
        try:
            response = gemini_model.generate_content(prompt)
            print(f'   {response.text.strip()[:300]}')
            continue
        except Exception:
            pass

    print('   (No AI available — try this when Ollama or Gemini is set up)')

print('\n💡 Same question, completely different perspectives — just by changing the role!')

🎭 EXPERIMENT 1: How Role Changes the Answer
Question: Why did so many people die on the Titanic?

🎭 Role: You are a maritime safety engineer.
   The tragic sinking of the Titanic was a complex event with multiple contributing factors, but some key reasons include the catastrophic failure of its watertight compartments due to excessive speed and inadequate design, as well as a combination of human error and regulatory oversights. The ship's h

🎭 Role: You are a social historian studying class inequality.
   The tragic loss of life on the Titanic is a stark example of class inequality and its devastating consequences. The ship's catastrophic sinking was not just a result of engineering failures, but also of the social and economic disparities that existed among its passengers. The fact that over 60% of 

🎭 Role: You are a 10-year-old explaining to a friend.
   So, like, the Titanic was this huge ship that sank in the ocean because it hit an iceberg! The problem was, there weren't enough 

In [ ]:
# --- Experiment 2: Format Controls Output ---
# Same content, different output formats

base = 'Explain the key factors that determined Titanic survival.'

formats = [
    ('Bullet points', f'{base} Use bullet points.'),
    ('Haiku',         f'{base} Answer as a haiku (5-7-5 syllables).'),
    ('Tweet',         f'{base} Answer in a single tweet (under 280 characters).'),
    ('ELI5',          f'{base} Explain like I am 5 years old.'),
]

print('📝 EXPERIMENT 2: Format Controls Output')
print('=' * 60)

for label, prompt in formats:
    print(f'\n📝 Format: {label}')

    if ollama_ready:
        try:
            resp = requests.post(
                f'{OLLAMA_URL}/api/generate',
                json={'model': ollama_model, 'prompt': prompt, 'stream': False, 'options': {'temperature': 0.5}},
                timeout=30
            )
            if resp.status_code == 200:
                print(f'   {resp.json().get("response", "").strip()[:300]}')
                continue
        except Exception:
            pass

    if gemini_ready:
        try:
            response = gemini_model.generate_content(prompt)
            print(f'   {response.text.strip()[:300]}')
            continue
        except Exception:
            pass

    print('   (No AI available — try this when Ollama or Gemini is set up)')

print('\n💡 The format instruction completely changes how the AI structures its answer!')

📝 EXPERIMENT 2: Format Controls Output

📝 Format: Bullet points
   Here are the key factors that determined Titanic survival:

• **Class and social status**: First-class passengers had a significantly higher chance of survival due to their ability to afford lifeboats, access to crew knowledge, and priority boarding procedures.

• **Sex and age**: Women and children

📝 Format: Haiku
   Class and luck prevail
First-class passengers safe
Fate's cruel design

📝 Format: Tweet
   Did you know? Key factors determining #TitanicSurvival included: 1) Having a cabin on a lower deck, 2) Being female or traveling with family, 3) Having access to lifeboats via the boat deck, and 4) Being in first class. Class & location mattered most! #Titanic

📝 Format: ELI5
   Let me tell you a story about the Titanic!

So, there was a big ship called the Titanic. It was like a giant toy boat, but it was real! And it had people on board who were going to have a fun trip across the ocean.

But sometimes, bad things

In [ ]:
# --- Experiment 3: Temperature = Creativity Dial ---
# Low temperature = focused, deterministic
# High temperature = creative, varied

prompt = 'Describe what it was like on the Titanic the night it sank. One paragraph.'

temperatures = [0.0, 0.5, 1.0]

print('🌡️ EXPERIMENT 3: Temperature = Creativity Dial')
print(f'Prompt: {prompt}')
print('=' * 60)

for temp in temperatures:
    label = {0.0: 'Focused/Factual', 0.5: 'Balanced', 1.0: 'Creative/Varied'}[temp]
    print(f'\n🌡️ Temperature: {temp} ({label})')

    if ollama_ready:
        try:
            resp = requests.post(
                f'{OLLAMA_URL}/api/generate',
                json={'model': ollama_model, 'prompt': prompt, 'stream': False, 'options': {'temperature': temp}},
                timeout=30
            )
            if resp.status_code == 200:
                print(f'   {resp.json().get("response", "").strip()[:400]}')
                continue
        except Exception:
            pass

    if gemini_ready:
        try:
            response = gemini_model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(temperature=temp)
            )
            print(f'   {response.text.strip()[:400]}')
            continue
        except Exception:
            pass

    print('   (No AI available — try this when Ollama or Gemini is set up)')

print('\n💡 Temperature 0 = safe and predictable. Temperature 1 = creative and surprising.')
print('   For factual tasks, use low temperature. For creative tasks, go higher.')

🌡️ EXPERIMENT 3: Temperature = Creativity Dial
Prompt: Describe what it was like on the Titanic the night it sank. One paragraph.

🌡️ Temperature: 0.0 (Focused/Factual)
   The night of April 14, 1912, was a chaotic and eerie evening aboard the RMS Titanic as it navigated through the icy waters of the North Atlantic. The ship's grand staircase, opulent dining rooms, and lavish ballrooms were filled with the sounds of laughter, music, and champagne toasts, as the wealthy and elite celebrated the vessel's maiden voyage from Southampton to New York. However, beneath the

🌡️ Temperature: 0.5 (Balanced)
   On that fateful night, April 14, 1912, the RMS Titanic was a ship of opulence and grandeur, but also one of hubris and complacency. As the vessel navigated through the dark waters of the North Atlantic, the crew and passengers were oblivious to the danger lurking beneath the surface. The air was thick with the sound of laughter, music, and champagne toasts in the ship's luxurious saloons a

### 🚀 Build Your Own Prompt

Use the cell below to write your own prompt. Mix and match:
- **Role**: Who should the AI be?
- **Context**: What background info does it need?
- **Task**: What should it do?
- **Format**: How should it respond?
- **Temperature**: How creative should it be?

In [ ]:
# --- Your Custom Prompt ---
# Edit the prompt below and run this cell!

my_prompt = """You are a poet who loves history.

Write a short poem about the Titanic's last night.
Keep it under 8 lines."""

print(chat(my_prompt))

🦙 Ollama: On the fateful eve, so calm and bright,
The Titanic sailed, with hope in sight.
But beneath the waves, a doom did lie,
A disaster waiting, to say goodbye.
The ship of dreams, now lost at sea,
A night that would be etched in history.
The cries of those who perished, still echo free.


## Key Takeaways & Workshop Closing

---

### The Two Worlds of AI

| | Predictive AI | Generative AI |
|---|---|---|
| **Input** | Structured data (rows & columns) | Natural language (prompts) |
| **Output** | Categories or numbers | Text, code, images, reasoning |
| **Training** | You train the model on your data | Pre-trained on massive datasets |
| **Skill needed** | Feature engineering, statistics | Prompt engineering, evaluation |
| **Strength** | Precise, measurable, auditable | Flexible, creative, conversational |
| **Limitation** | Can only predict what it was trained on | Can hallucinate, hard to verify |
| **Best for** | Classification, regression, forecasting | Content creation, analysis, chat |

**Neither is better --- they solve different problems.** The best projects combine both.

### What You Learned Today

1. **Predictive AI** follows a pipeline: Load > Clean > Train > Predict
2. **Generative AI** follows a conversation: Prompt > Generate > Evaluate > Refine
3. **Prompt engineering** is the new programming --- role, context, task, and format matter
4. **Local AI** (Ollama on Ubuntu) gives you privacy and control
5. **Cloud AI** (Gemini) gives you power and convenience
6. **The fallback pattern** (Data > AI > Rules) makes your apps reliable
7. **Temperature** controls creativity vs consistency

### Your AI Toolkit

**Local-first (Ubuntu-friendly):**
- [Ollama](https://ollama.ai/) --- Run LLMs locally with one command
- [LM Studio](https://lmstudio.ai/) --- GUI for local models
- [Hugging Face](https://huggingface.co/) --- Open-source model hub

**Cloud APIs:**
- [Google AI Studio](https://makersuite.google.com/) --- Free Gemini API key
- [OpenAI Platform](https://platform.openai.com/) --- GPT models

**Learning resources:**
- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [Fast.ai Practical Deep Learning](https://course.fast.ai/)
- [scikit-learn Documentation](https://scikit-learn.org/)

### Choosing the Right AI for Your Project

Ask yourself:

```
Do I have labeled data + need a specific prediction?
  -> YES -> Predictive AI (scikit-learn, TensorFlow)
  -> NO  -> Do I need to generate, explain, or converse?
            -> YES -> Generative AI (Ollama, Gemini, GPT)
            -> BOTH? -> Combine them! Predict first, then explain with GenAI
```

**The winning pattern**: Use Predictive AI for the numbers, Generative AI for the narrative.

In [ ]:
# --- Final Demo: The Combined Power ---
# Predictive AI gave us: 78.5% accuracy on survival prediction
# Generative AI gives us: understanding, explanation, creativity

final_prompt = (
    'Based on Titanic survival data:\n'
    '- Women survived at 74%, men at 19%\n'
    '- 1st class: 63%, 2nd class: 47%, 3rd class: 24%\n'
    '- Children under 10 had higher survival rates\n\n'
    'In 3 bullet points, explain what this tells us about '
    'human behavior during disasters. Then suggest one way '
    'modern safety systems have improved because of this.'
)

print('COMBINING BOTH WORLDS')
print('Predictive AI found the patterns. Generative AI explains them.')
print('=' * 60)
print(chat(final_prompt))

COMBINING BOTH WORLDS
Predictive AI found the patterns. Generative AI explains them.
📊 REAL DATA: Women had 74.2% survival vs men at 18.9%. The "women and children first" policy was clearly enforced.


### What's Next?

You now have everything you need to:

1. **Build a Predictive AI pipeline** for any classification problem
2. **Integrate Generative AI** into your apps via Ollama or Gemini
3. **Engineer effective prompts** using role, context, task, and format
4. **Choose the right approach** for your next project or hackathon

---

*From Predictive to Generative AI --- UbuCon 2026*

**Thank you! Now go build something amazing.**